# Error Analysis — Thai Job NER v3 (F1=0.897)

Analyze model errors on the test set to identify failure modes, per-entity regressions, and improvement opportunities.

**Model:** WangchanBERTa fine-tuned v3 (1,253 posts, class-weighted loss, label smoothing)  
**Test set:** 126 examples

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Use a font that supports Thai characters
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'Tahoma', 'sans-serif']
matplotlib.rcParams['figure.figsize'] = (12, 6)

sys.path.insert(0, str(Path.cwd().parent))

# Load pre-computed evaluation results
with open("../results_v3/per_entity_test.json") as f:
    v3_results = json.load(f)

print("v3 results loaded successfully")
print(f"Overall F1: {v3_results['overall']['f1']:.3f}")
print(f"Overall Precision: {v3_results['overall']['precision']:.3f}")
print(f"Overall Recall: {v3_results['overall']['recall']:.3f}")

## 1. Per-Entity F1 Comparison (v2 → v3)

In [ ]:
# v2 baseline results for comparison
v2_f1 = {
    "HARD_SKILL": 0.761,
    "PERSON": 0.892,
    "LOCATION": 0.861,
    "COMPENSATION": 0.819,
    "EMPLOYMENT_TERMS": 0.850,
    "CONTACT": 0.957,
    "DEMOGRAPHIC": 0.776,
}

# Extract v3 F1
entities = list(v2_f1.keys())
v2_scores = [v2_f1[e] for e in entities]
v3_scores = [v3_results["per_entity"][e]["f1"] for e in entities]
deltas = [v3 - v2 for v2, v3 in zip(v2_scores, v3_scores)]

# Sort by delta (biggest improvement first)
sorted_idx = np.argsort(deltas)[::-1]
entities_sorted = [entities[i] for i in sorted_idx]
v2_sorted = [v2_scores[i] for i in sorted_idx]
v3_sorted = [v3_scores[i] for i in sorted_idx]
deltas_sorted = [deltas[i] for i in sorted_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: v2 vs v3 F1
x = np.arange(len(entities_sorted))
width = 0.35
bars1 = ax1.bar(x - width/2, v2_sorted, width, label='v2 (F1=0.828)', color='#93c5fd', edgecolor='white')
bars2 = ax1.bar(x + width/2, v3_sorted, width, label='v3 (F1=0.897)', color='#3b82f6', edgecolor='white')
ax1.set_ylabel('F1 Score')
ax1.set_title('Per-Entity F1: v2 vs v3')
ax1.set_xticks(x)
ax1.set_xticklabels(entities_sorted, rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0.6, 1.0)
ax1.axhline(y=0.85, color='red', linestyle='--', alpha=0.5, label='Target (0.85)')

# Delta chart
colors = ['#22c55e' if d >= 0 else '#ef4444' for d in deltas_sorted]
bars3 = ax2.bar(entities_sorted, [d * 100 for d in deltas_sorted], color=colors, edgecolor='white')
ax2.set_ylabel('F1 Change (percentage points)')
ax2.set_title('v2 → v3 Improvement')
ax2.set_xticklabels(entities_sorted, rotation=45, ha='right')
ax2.axhline(y=0, color='black', linewidth=0.5)

# Add value labels
for bar, delta in zip(bars3, deltas_sorted):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + (0.3 if delta >= 0 else -0.8),
             f'{delta*100:+.1f}pp', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../results_v3/per_entity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary:")
for e, v2, v3, d in zip(entities_sorted, v2_sorted, v3_sorted, deltas_sorted):
    marker = "✅" if d >= 0 else "❌"
    print(f"  {marker} {e:20s}  {v2:.3f} → {v3:.3f}  ({d*100:+.1f}pp)")

## 2. Confusion Matrix Heatmap

Token-level confusion matrix showing which entity types the model confuses. Rows = true labels, columns = predicted labels.

In [ ]:
confusion = v3_results["confusion_matrix"]

# Entity types in display order (exclude O for entity-focused view)
entity_types = ["HARD_SKILL", "PERSON", "LOCATION", "COMPENSATION",
                "EMPLOYMENT_TERMS", "CONTACT", "DEMOGRAPHIC"]
all_types = entity_types + ["O"]

# Build matrix
matrix = np.zeros((len(all_types), len(all_types)), dtype=int)
for i, true_type in enumerate(all_types):
    for j, pred_type in enumerate(all_types):
        matrix[i, j] = confusion.get(true_type, {}).get(pred_type, 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Full confusion matrix
im1 = ax1.imshow(matrix, cmap='Blues', aspect='auto')
ax1.set_xticks(range(len(all_types)))
ax1.set_yticks(range(len(all_types)))
ax1.set_xticklabels(all_types, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(all_types, fontsize=9)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_title('Full Confusion Matrix (token-level)')

# Add text annotations
for i in range(len(all_types)):
    for j in range(len(all_types)):
        val = matrix[i, j]
        if val > 0:
            color = 'white' if val > matrix.max() * 0.5 else 'black'
            ax1.text(j, i, str(val), ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im1, ax=ax1, shrink=0.8)

# Error-only matrix (exclude diagonal) — focus on misclassifications
error_matrix = matrix.copy()
np.fill_diagonal(error_matrix, 0)

im2 = ax2.imshow(error_matrix, cmap='Reds', aspect='auto')
ax2.set_xticks(range(len(all_types)))
ax2.set_yticks(range(len(all_types)))
ax2.set_xticklabels(all_types, rotation=45, ha='right', fontsize=9)
ax2.set_yticklabels(all_types, fontsize=9)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_title('Errors Only (diagonal removed)')

for i in range(len(all_types)):
    for j in range(len(all_types)):
        val = error_matrix[i, j]
        if val > 0:
            color = 'white' if val > error_matrix.max() * 0.5 else 'black'
            ax2.text(j, i, str(val), ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.savefig('../results_v3/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top error pairs
print("\nTop error pairs (true → predicted, count):")
errors = []
for i, true_type in enumerate(all_types):
    for j, pred_type in enumerate(all_types):
        if i != j and error_matrix[i, j] > 0:
            errors.append((true_type, pred_type, error_matrix[i, j]))
errors.sort(key=lambda x: -x[2])
for true_t, pred_t, count in errors[:10]:
    print(f"  {true_t:20s} → {pred_t:20s}  {count:>4d} tokens")

## 3. Precision vs Recall by Entity

Shows whether each entity type suffers more from false positives (low precision) or missed entities (low recall).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for entity in entity_types:
    data = v3_results["per_entity"][entity]
    p, r, f1 = data["precision"], data["recall"], data["f1"]
    size = data["number"] * 2  # Scale point size by support

    ax.scatter(r, p, s=size, alpha=0.8, zorder=5)
    ax.annotate(f'{entity}\nF1={f1:.3f}', (r, p),
                textcoords="offset points", xytext=(10, 5), fontsize=8)

# Add F1 isocurves
for f1_val in [0.7, 0.8, 0.9, 0.95]:
    r_range = np.linspace(0.5, 1.0, 100)
    p_vals = (f1_val * r_range) / (2 * r_range - f1_val)
    valid = (p_vals > 0.5) & (p_vals <= 1.0)
    ax.plot(r_range[valid], p_vals[valid], '--', alpha=0.3, color='gray')
    # Label the curve
    idx = np.argmin(np.abs(p_vals - r_range))
    if valid[idx]:
        ax.text(r_range[idx], p_vals[idx], f'F1={f1_val}', fontsize=7, alpha=0.5)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision vs Recall by Entity Type (v3)\n(point size ∝ support count)')
ax.set_xlim(0.65, 1.02)
ax.set_ylim(0.62, 1.02)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../results_v3/precision_recall_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key observations:")
print("• COMPENSATION: High recall (0.884) but low precision (0.673) → too many false positives")
print("• LOCATION: Near-perfect on both axes (P=0.928, R=0.991)")
print("• PERSON: Highest recall (0.958) but precision could improve (0.861)")

## 4. Error Examples — Model Predictions on Test Data

Run the model on test examples and find posts where it makes mistakes. Focus on COMPENSATION false positives (biggest error source).

In [ ]:
import torch
from datasets import DatasetDict
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
from src.alignment.token_mapper import DEFAULT_LABELS, IGNORE_INDEX

# Load model and dataset
model_dir = "../results_v3/final"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForTokenClassification.from_pretrained(model_dir)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
model.eval()

dataset = DatasetDict.load_from_disk("../data/processed")
test_data = dataset["test"]

print(f"Model loaded on {device}")
print(f"Test set: {len(test_data)} examples")
print(f"Labels: {DEFAULT_LABELS}")

In [ ]:
def get_predictions(example):
    """Run model on a single example and return pred/true tag sequences."""
    input_ids = torch.tensor([example["input_ids"]], device=device)
    attention_mask = torch.tensor([example["attention_mask"]], device=device)
    labels = example["labels"]

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
    logits = outputs.logits[0]
    probs = torch.softmax(logits, dim=-1)
    pred_ids = torch.argmax(probs, dim=-1).cpu().tolist()
    max_probs = probs.max(dim=-1).values.cpu().tolist()
    
    # Decode tokens and labels, skipping padding
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
    results = []
    for i, (token, pred_id, label_id, prob) in enumerate(zip(tokens, pred_ids, labels, max_probs)):
        if label_id == IGNORE_INDEX:
            continue
        pred_label = DEFAULT_LABELS[pred_id]
        true_label = DEFAULT_LABELS[label_id]
        is_error = pred_label != true_label
        results.append({
            "token": token,
            "true": true_label,
            "pred": pred_label,
            "conf": prob,
            "error": is_error,
        })
    return results


def count_errors(results):
    """Count errors by category."""
    errors = defaultdict(int)
    for r in results:
        if r["error"]:
            true_base = r["true"].split("-", 1)[1] if "-" in r["true"] else r["true"]
            pred_base = r["pred"].split("-", 1)[1] if "-" in r["pred"] else r["pred"]
            errors[f"{true_base}→{pred_base}"] += 1
    return dict(errors)


# Run predictions on all test examples
all_errors_by_example = []
for idx in range(len(test_data)):
    example = test_data[idx]
    results = get_predictions(example)
    errors = count_errors(results)
    total_errors = sum(errors.values())
    all_errors_by_example.append({
        "idx": idx,
        "results": results,
        "errors": errors,
        "total_errors": total_errors,
        "total_tokens": len(results),
    })

# Sort by most errors
all_errors_by_example.sort(key=lambda x: -x["total_errors"])

print(f"Evaluated {len(test_data)} test examples")
print(f"Examples with errors: {sum(1 for e in all_errors_by_example if e['total_errors'] > 0)}")
print(f"Error-free examples: {sum(1 for e in all_errors_by_example if e['total_errors'] == 0)}")
print(f"\nTop 5 most error-prone examples:")
for e in all_errors_by_example[:5]:
    print(f"  Example {e['idx']}: {e['total_errors']} errors / {e['total_tokens']} tokens — {e['errors']}")

## 5. Detailed Error Inspection

Show the worst predictions token-by-token. Red highlights indicate misclassified tokens.

In [ ]:
def display_example(example_data, max_tokens=80):
    """Display token-level predictions for one example, highlighting errors."""
    results = example_data["results"][:max_tokens]
    idx = example_data["idx"]
    
    print(f"{'='*80}")
    print(f"Example {idx} — {example_data['total_errors']} errors / {example_data['total_tokens']} tokens")
    print(f"Error types: {example_data['errors']}")
    print(f"{'='*80}")
    
    # Reconstruct text
    text_tokens = [r["token"].replace("▁", " ") for r in results]
    print(f"Text: {''.join(text_tokens)[:200]}...")
    print()
    
    print(f"{'Token':<20} {'True':<22} {'Predicted':<22} {'Conf':>6} {'':>4}")
    print("-" * 80)
    
    for r in results:
        token_display = r["token"].replace("▁", "_")[:18]
        marker = " <<< ERROR" if r["error"] else ""
        conf = f"{r['conf']:.3f}"
        
        if r["error"]:
            print(f"{token_display:<20} {r['true']:<22} {r['pred']:<22} {conf:>6}{marker}")
        elif r["true"] != "O":
            # Show correctly tagged entities too
            print(f"{token_display:<20} {r['true']:<22} {r['pred']:<22} {conf:>6}")
    print()

# Show top 5 worst examples
for example_data in all_errors_by_example[:5]:
    display_example(example_data)

## 6. Error Distribution by Category

Categorize all errors into failure modes: false positives (O→entity), missed entities (entity→O), and cross-entity confusion (entity→wrong entity).

In [ ]:
# Categorize errors from confusion matrix
false_positives = {}  # O predicted as entity
missed_entities = {}  # entity predicted as O
cross_confusion = {}  # entity predicted as wrong entity

cm = v3_results["confusion_matrix"]

# False positives: O row
for pred_type, count in cm.get("O", {}).items():
    if pred_type != "O":
        false_positives[pred_type] = count

# Missed entities: entity rows, O column
for true_type, preds in cm.items():
    if true_type != "O":
        if "O" in preds:
            missed_entities[true_type] = preds["O"]

# Cross-entity confusion: entity→entity (not O, not diagonal)
for true_type, preds in cm.items():
    if true_type == "O":
        continue
    for pred_type, count in preds.items():
        if pred_type != "O" and pred_type != true_type:
            cross_confusion[f"{true_type}→{pred_type}"] = count

total_fp = sum(false_positives.values())
total_missed = sum(missed_entities.values())
total_cross = sum(cross_confusion.values())
total_errors = total_fp + total_missed + total_cross

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# False positives
ax = axes[0]
fp_sorted = sorted(false_positives.items(), key=lambda x: -x[1])
ax.barh([x[0] for x in fp_sorted], [x[1] for x in fp_sorted], color='#ef4444')
ax.set_title(f'False Positives (O→Entity)\n{total_fp} tokens ({total_fp/total_errors*100:.0f}% of errors)')
ax.set_xlabel('Token count')
for i, (name, val) in enumerate(fp_sorted):
    ax.text(val + 0.5, i, str(val), va='center', fontsize=9)

# Missed entities
ax = axes[1]
me_sorted = sorted(missed_entities.items(), key=lambda x: -x[1])
ax.barh([x[0] for x in me_sorted], [x[1] for x in me_sorted], color='#f97316')
ax.set_title(f'Missed Entities (Entity→O)\n{total_missed} tokens ({total_missed/total_errors*100:.0f}% of errors)')
ax.set_xlabel('Token count')
for i, (name, val) in enumerate(me_sorted):
    ax.text(val + 0.3, i, str(val), va='center', fontsize=9)

# Cross-entity confusion
ax = axes[2]
cc_sorted = sorted(cross_confusion.items(), key=lambda x: -x[1])[:8]
ax.barh([x[0] for x in cc_sorted], [x[1] for x in cc_sorted], color='#eab308')
ax.set_title(f'Cross-Entity Confusion\n{total_cross} tokens ({total_cross/total_errors*100:.0f}% of errors)')
ax.set_xlabel('Token count')
for i, (name, val) in enumerate(cc_sorted):
    ax.text(val + 0.2, i, str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results_v3/error_categories.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nError budget:")
print(f"  False positives (O→Entity):  {total_fp:>4} tokens ({total_fp/total_errors*100:.1f}%)")
print(f"  Missed entities (Entity→O):  {total_missed:>4} tokens ({total_missed/total_errors*100:.1f}%)")
print(f"  Cross-entity confusion:      {total_cross:>4} tokens ({total_cross/total_errors*100:.1f}%)")
print(f"  Total errors:                {total_errors:>4} tokens")

## 7. Confidence Distribution for Correct vs Incorrect Predictions

Are errors low-confidence? If so, a confidence threshold could filter out mistakes.

In [ ]:
# Collect confidence scores for correct vs incorrect entity predictions
correct_confs = []
error_confs = []

for example_data in all_errors_by_example:
    for r in example_data["results"]:
        # Only look at tokens where either true or pred is an entity (skip O→O)
        if r["true"] == "O" and r["pred"] == "O":
            continue
        if r["error"]:
            error_confs.append(r["conf"])
        else:
            correct_confs.append(r["conf"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
bins = np.linspace(0.3, 1.0, 30)
ax1.hist(correct_confs, bins=bins, alpha=0.7, label=f'Correct ({len(correct_confs)})', color='#22c55e')
ax1.hist(error_confs, bins=bins, alpha=0.7, label=f'Errors ({len(error_confs)})', color='#ef4444')
ax1.set_xlabel('Model Confidence')
ax1.set_ylabel('Token Count')
ax1.set_title('Confidence Distribution: Correct vs Error')
ax1.legend()
ax1.axvline(x=0.8, color='black', linestyle='--', alpha=0.3, label='Threshold=0.8')

# CDF — what fraction of errors can we catch at each threshold?
error_sorted = np.sort(error_confs)
correct_sorted = np.sort(correct_confs)

thresholds = np.linspace(0.5, 1.0, 50)
error_rejected = [np.mean(np.array(error_confs) < t) for t in thresholds]
correct_rejected = [np.mean(np.array(correct_confs) < t) for t in thresholds]

ax2.plot(thresholds, error_rejected, 'r-', label='Errors rejected', linewidth=2)
ax2.plot(thresholds, correct_rejected, 'g-', label='Correct rejected (cost)', linewidth=2)
ax2.set_xlabel('Confidence Threshold')
ax2.set_ylabel('Fraction Rejected')
ax2.set_title('Confidence Filtering Trade-off')
ax2.legend()
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../results_v3/confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Correct entity tokens: {len(correct_confs)}, mean conf: {np.mean(correct_confs):.3f}")
print(f"Error entity tokens:   {len(error_confs)}, mean conf: {np.mean(error_confs):.3f}")
print(f"\nAt threshold=0.8: reject {np.mean(np.array(error_confs) < 0.8)*100:.1f}% of errors, "
      f"lose {np.mean(np.array(correct_confs) < 0.8)*100:.1f}% of correct")

## 8. Summary & Recommendations

### Key Findings

1. **Overall: F1 improved 0.828 → 0.897 (+6.9pp)** through data scaling (758→1,253 posts) + class-weighted loss + label smoothing

2. **Biggest improvements:** HARD_SKILL (+14.2pp), LOCATION (+9.8pp), DEMOGRAPHIC (+9.9pp) — all benefited from class weighting pushing the model to attend more to rare entities

3. **One regression: COMPENSATION (-5.5pp)** — class weight (4.11) made the model too eager to tag numbers as salary. 33 O tokens falsely predicted as COMPENSATION

4. **Error budget:** False positives (O→Entity) dominate errors. The model over-predicts more than it misses.

5. **Cross-entity confusion is rare** — only DEMOGRAPHIC↔HARD_SKILL shows meaningful confusion (11 tokens)

### Recommendations for v4

| Priority | Action | Expected Impact |
|----------|--------|-----------------|
| P0 | Lower COMPENSATION class weight to ~2.0 | Fix precision regression |
| P0 | Add hard negatives: posts with non-salary numbers (ages, room numbers, quantities) | Reduce COMPENSATION false positives |
| P1 | Add more PERSON examples without Thai honorifics | Improve PERSON recall |
| P2 | Post-processing: require currency word (บาท/เงิน) near COMPENSATION spans | Filter false positives at inference |
| P3 | Confidence threshold at inference (~0.8) | Trade-off: reject uncertain predictions |